<a href="https://colab.research.google.com/github/PattiAlves/Modelagem_IA/blob/main/Notebooks/4.%20CORRELA%C3%87%C3%83O%20DE%20SPEARMAN%20E%20AN%C3%81LISE%20DE%20RESULTADOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **4.CORRELAÇÃO DE SPEARMAN E ANÁLISE DE RESULTADOS**

### **4.1 Correlação de Spearman entre dados da lagoa reconstruída, maré e chuva**

In [1]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_reconst = os.path.join(pasta_consolidados, 'SERIE_LAGOA_RECONST')
pasta_eda = os.path.join(pasta_reconst, 'ANALISE_EXPLORATORIA')
os.makedirs(pasta_eda, exist_ok=True)

caminho_chuva_mare = os.path.join(
    pasta_consolidados, 'chuva_mare_10min_092017_092026.csv'
)

arquivos_reconst = sorted(
    glob.glob(os.path.join(pasta_reconst, 'serie_lagoa_reconstituida_*.csv'))
)
if not arquivos_reconst:
  raise FileNotFoundError(
      f'Nenhum arquivo reconstruído encontrado em {pasta_reconst}'
  )
caminho_reconst = arquivos_reconst[-1]

# 2. Carregar série reconstruída e planilha chuva_mare na grade de 10 min (sem interpolar)
df_rec = pd.read_csv(caminho_reconst)
col_dt_rec = 'Date_time' if 'Date_time' in df_rec.columns else df_rec.columns[0]
col_na_rec = 'NA_lagoa' if 'NA_lagoa' in df_rec.columns else df_rec.columns[1]

df_rec['dt'] = pd.to_datetime(df_rec[col_dt_rec], errors='coerce')
df_rec = (
    df_rec.dropna(subset=['dt'])
    .sort_values('dt')
    .drop_duplicates(subset=['dt'], keep='last')
    .set_index('dt')
)

df_cm = pd.read_csv(caminho_chuva_mare)
col_dt_cm = 'Date_time' if 'Date_time' in df_cm.columns else df_cm.columns[0]
df_cm['dt'] = pd.to_datetime(df_cm[col_dt_cm], errors='coerce')
df_cm = (
    df_cm.dropna(subset=['dt'])
    .sort_values('dt')
    .drop_duplicates(subset=['dt'], keep='last')
    .set_index('dt')
)

grade_10min = pd.date_range(
    start=min(df_rec.index.min(), df_cm.index.min()),
    end=max(df_rec.index.max(), df_cm.index.max()),
    freq='10min',
)

s_lagoa = pd.to_numeric(df_rec[col_na_rec], errors='coerce').reindex(grade_10min)
df_vars = df_cm.drop(columns=[col_dt_cm], errors='ignore').reindex(grade_10min)
for c in df_vars.columns:
  df_vars[c] = pd.to_numeric(df_vars[c], errors='coerce')


def calcular_spearman_tabela(s_alvo, df_preditores):
  linhas = []
  for col in df_preditores.columns:
    par = pd.concat([s_alvo, df_preditores[col]], axis=1).dropna()
    n_pares = len(par)
    if (
        n_pares > 2
        and par.iloc[:, 0].nunique() > 1
        and par.iloc[:, 1].nunique() > 1
    ):
      rho, p_val = spearmanr(par.iloc[:, 0], par.iloc[:, 1])
    else:
      rho, p_val = np.nan, np.nan
    linhas.append({
        'variavel': col,
        'n_pares_validos': n_pares,
        'spearman_rho': round(float(rho), 4) if pd.notna(rho) else np.nan,
        'abs_rho': round(abs(float(rho)), 4) if pd.notna(rho) else np.nan,
        'p_valor': p_val,
    })
  return pd.DataFrame(linhas).sort_values(
      by='abs_rho', ascending=False, na_position='last'
  )


# 3. Spearman para todas as variáveis da planilha consolidada
df_sp_todas = calcular_spearman_tabela(s_lagoa, df_vars)
df_sp_todas.to_csv(
    os.path.join(pasta_eda, 'spearman_todas_variaveis.csv'), index=False
)

print('=== TOP 15 CORRELAÇÕES DE SPEARMAN COM NA_LAGOA ===')
print(
    df_sp_todas[['variavel', 'n_pares_validos', 'spearman_rho', 'p_valor']]
    .head(15)
    .to_string(index=False)
)

# 4. Lags estendidos de maré (-3h a +6h), se houver coluna de maré instantânea
col_mare_base = (
    'mare_10min_m'
    if 'mare_10min_m' in df_vars.columns
    else (
        'nivel_mar_total_m' if 'nivel_mar_total_m' in df_vars.columns else None
    )
)

df_lags_mare = pd.DataFrame()
if col_mare_base is not None:
  # shift > 0: maré ocorrida há k passos (maré antecedendo o nível da lagoa)
  # shift < 0: maré futura em k passos (para checar assimetria / propagação)
  passos_lag = list(range(-36, 37, 3))  # de -6h a +6h a cada 30 min
  linhas_lm = []
  for p in passos_lag:
    s_m_lag = df_vars[col_mare_base].shift(p)
    par = pd.concat([s_lagoa, s_m_lag], axis=1).dropna()
    rho, p_val = (
        spearmanr(par.iloc[:, 0], par.iloc[:, 1])
        if len(par) > 2
        else (np.nan, np.nan)
    )
    linhas_lm.append({
        'lag_passos_10min': p,
        'lag_horas': round(p * 10 / 60.0, 2),
        'n_pares_validos': len(par),
        'spearman_rho': round(float(rho), 4) if pd.notna(rho) else np.nan,
        'p_valor': p_val,
    })
  df_lags_mare = pd.DataFrame(linhas_lm)
  df_lags_mare.to_csv(
      os.path.join(pasta_eda, 'spearman_lags_mare.csv'), index=False
  )

# 5. Spearman em regime com chuva vs. sem chuva (usando chuva_acum_6h_mm ou chuva_10min_mm)
col_ref_chuva = (
    'chuva_acum_6h_mm'
    if 'chuva_acum_6h_mm' in df_vars.columns
    else ('chuva_10min_mm' if 'chuva_10min_mm' in df_vars.columns else None)
)

if col_ref_chuva is not None:
  mask_com_chuva = df_vars[col_ref_chuva] > 0
  mask_sem_chuva = df_vars[col_ref_chuva] == 0

  sp_com_chuva = calcular_spearman_tabela(
      s_lagoa[mask_com_chuva], df_vars.loc[mask_com_chuva]
  ).rename(
      columns={
          'spearman_rho': 'rho_com_chuva',
          'n_pares_validos': 'n_com_chuva',
      }
  )
  sp_sem_chuva = calcular_spearman_tabela(
      s_lagoa[mask_sem_chuva], df_vars.loc[mask_sem_chuva]
  ).rename(
      columns={
          'spearman_rho': 'rho_sem_chuva',
          'n_pares_validos': 'n_sem_chuva',
      }
  )

  df_regimes = sp_com_chuva[['variavel', 'n_com_chuva', 'rho_com_chuva']].merge(
      sp_sem_chuva[['variavel', 'n_sem_chuva', 'rho_sem_chuva']],
      on='variavel',
      how='outer',
  )
  df_regimes.to_csv(
      os.path.join(pasta_eda, 'spearman_regimes_chuva.csv'), index=False
  )
  print(
      f'\n=== SPEARMAN POR REGIME ({col_ref_chuva} > 0 vs. == 0) — TOP 10 ==='
  )
  print(df_regimes.head(10).to_string(index=False))

# 6. Gráficos
plt.rcParams['figure.dpi'] = 140

# 6a. Ranking horizontal de Spearman (top 18 variáveis)
top_plot = df_sp_todas.dropna(subset=['spearman_rho']).head(18).iloc[::-1]
cores = ['#1f77b4' if v >= 0 else '#d95f02' for v in top_plot['spearman_rho']]

fig, ax = plt.subplots(figsize=(10.5, 5.5))
barras = ax.barh(
    top_plot['variavel'],
    top_plot['spearman_rho'],
    color=cores,
    edgecolor='black',
    linewidth=0.5,
)
ax.axvline(0, color='black', linewidth=0.9)
for b in barras:
  w = b.get_width()
  desloc = 0.008 if w >= 0 else -0.008
  alinh = 'left' if w >= 0 else 'right'
  ax.annotate(
      f'{w:+.3f}',
      (w, b.get_y() + b.get_height() / 2),
      xytext=(desloc, 0),
      textcoords='offset points',
      ha=alinh,
      va='center',
      fontsize=8,
  )

ax.set_xlabel('Correlação de Spearman (ρ) com NA_lagoa')
ax.set_title('Ranking de Correlação de Spearman — NA_lagoa vs. Variáveis')
ax.grid(True, axis='x', alpha=0.25)
plt.tight_layout()
caminho_fig1 = os.path.join(pasta_eda, 'spearman_ranking_variaveis.png')
plt.savefig(caminho_fig1, bbox_inches='tight')
plt.close()

# 6b. Acumulados de chuva e lags de maré (se disponíveis)
cols_acum = [
    c
    for c in [
        'chuva_10min_mm',
        'chuva_acum_30min_mm',
        'chuva_acum_1h_mm',
        'chuva_acum_3h_mm',
        'chuva_acum_6h_mm',
        'chuva_acum_12h_mm',
        'chuva_acum_24h_mm',
    ]
    if c in df_vars.columns
]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

if cols_acum:
  sub_acum = df_sp_todas.set_index('variavel').reindex(cols_acum)
  axes[0].plot(
      range(len(cols_acum)),
      sub_acum['spearman_rho'].values,
      marker='o',
      color='#08519c',
      linewidth=2.0,
  )
  axes[0].set_xticks(range(len(cols_acum)))
  axes[0].set_xticklabels(
      [c.replace('chuva_', '').replace('_mm', '') for c in cols_acum],
      rotation=25,
  )
  axes[0].set_ylabel('Spearman (ρ)')
  axes[0].set_title('Spearman por Janela de Chuva')
  axes[0].grid(True, alpha=0.25)
else:
  axes[0].set_visible(False)

if not df_lags_mare.empty:
  axes[1].plot(
      df_lags_mare['lag_horas'],
      df_lags_mare['spearman_rho'],
      marker='s',
      color='#cb181d',
      linewidth=1.8,
  )
  axes[1].axvline(0, color='gray', linestyle='--', linewidth=0.9)
  axes[1].axhline(0, color='black', linewidth=0.8)
  axes[1].set_xlabel('Defasagem da maré (horas; + = maré antecede NA_lagoa)')
  axes[1].set_ylabel('Spearman (ρ)')
  axes[1].set_title('Spearman — Maré Defasada vs. NA_lagoa')
  axes[1].grid(True, alpha=0.25)
else:
  axes[1].set_visible(False)

plt.tight_layout()
caminho_fig2 = os.path.join(pasta_eda, 'spearman_chuva_acum_e_lags_mare.png')
plt.savefig(caminho_fig2, bbox_inches='tight')
plt.close()

print(f'\nResultados de Spearman salvos em: {pasta_eda}')

Mounted at /content/drive
=== TOP 15 CORRELAÇÕES DE SPEARMAN COM NA_LAGOA ===
           variavel  n_pares_validos  spearman_rho  p_valor
    mare_lag_360min           473499        0.4607      0.0
  chuva_acum_24h_mm           473499        0.3890      0.0
  chuva_acum_12h_mm           473499        0.3021      0.0
   chuva_acum_6h_mm           473499        0.2434      0.0
   chuva_acum_3h_mm           473499        0.2087      0.0
   chuva_lag_360min           473499        0.1922      0.0
   chuva_lag_180min           473499        0.1737      0.0
   chuva_acum_1h_mm           473499        0.1734      0.0
   chuva_lag_120min           473499        0.1640      0.0
    mare_lag_180min           473499        0.1609      0.0
chuva_acum_30min_mm           473499        0.1590      0.0
    chuva_lag_60min           473499        0.1536      0.0
    chuva_lag_30min           473499        0.1518      0.0
    chuva_lag_10min           473499        0.1483      0.0
     chuva_10min_mm   

### **4.2 Correlação de Spearman por regime de chuva (com chuva vs. sem chuva) e exploração de defasagens adicionais da maré**

In [2]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_reconst = os.path.join(pasta_consolidados, 'SERIE_LAGOA_RECONST')
pasta_eda = os.path.join(pasta_reconst, 'ANALISE_EXPLORATORIA')
os.makedirs(pasta_eda, exist_ok=True)

caminho_chuva_mare = os.path.join(
    pasta_consolidados, 'chuva_mare_10min_092017_092026.csv'
)

arquivos_reconst = sorted(
    glob.glob(os.path.join(pasta_reconst, 'serie_lagoa_reconstituida_*.csv'))
)
if not arquivos_reconst:
  raise FileNotFoundError(
      f'Nenhum arquivo reconstruído encontrado em {pasta_reconst}'
  )
caminho_reconst = arquivos_reconst[-1]

# 2. Carregar séries na grade regular de 10 min (sem interpolar)
df_rec = pd.read_csv(caminho_reconst)
col_dt_rec = 'Date_time' if 'Date_time' in df_rec.columns else df_rec.columns[0]
col_na_rec = 'NA_lagoa' if 'NA_lagoa' in df_rec.columns else df_rec.columns[1]

df_rec['dt'] = pd.to_datetime(df_rec[col_dt_rec], errors='coerce')
df_rec = (
    df_rec.dropna(subset=['dt'])
    .sort_values('dt')
    .drop_duplicates(subset=['dt'], keep='last')
    .set_index('dt')
)

df_cm = pd.read_csv(caminho_chuva_mare)
col_dt_cm = 'Date_time' if 'Date_time' in df_cm.columns else df_cm.columns[0]
df_cm['dt'] = pd.to_datetime(df_cm[col_dt_cm], errors='coerce')
df_cm = (
    df_cm.dropna(subset=['dt'])
    .sort_values('dt')
    .drop_duplicates(subset=['dt'], keep='last')
    .set_index('dt')
)

grade_10min = pd.date_range(
    start=min(df_rec.index.min(), df_cm.index.min()),
    end=max(df_rec.index.max(), df_cm.index.max()),
    freq='10min',
)

s_lagoa = pd.to_numeric(df_rec[col_na_rec], errors='coerce').reindex(grade_10min)
df_vars = df_cm.drop(columns=[col_dt_cm], errors='ignore').reindex(grade_10min)
for c in df_vars.columns:
  df_vars[c] = pd.to_numeric(df_vars[c], errors='coerce')


def calcular_spearman_subconjunto(s_alvo, df_preditores, mascara=None):
  if mascara is not None:
    s_a = s_alvo[mascara]
    df_p = df_preditores.loc[mascara]
  else:
    s_a = s_alvo
    df_p = df_preditores

  linhas = []
  for col in df_p.columns:
    par = pd.concat([s_a, df_p[col]], axis=1).dropna()
    n_pares = len(par)
    if (
        n_pares > 2
        and par.iloc[:, 0].nunique() > 1
        and par.iloc[:, 1].nunique() > 1
    ):
      rho, p_val = spearmanr(par.iloc[:, 0], par.iloc[:, 1])
    else:
      rho, p_val = np.nan, np.nan
    linhas.append({
        'variavel': col,
        'n_pares': n_pares,
        'rho': round(float(rho), 4) if pd.notna(rho) else np.nan,
        'p_valor': p_val,
    })
  return pd.DataFrame(linhas)


# 3. Spearman por regime: COM CHUVA vs. SEM CHUVA
col_ref_chuva = (
    'chuva_acum_6h_mm'
    if 'chuva_acum_6h_mm' in df_vars.columns
    else ('chuva_10min_mm' if 'chuva_10min_mm' in df_vars.columns else None)
)
if col_ref_chuva is None:
  raise ValueError(
      'Nenhuma coluna de chuva encontrada para definir os regimes.'
  )

mask_com_chuva = df_vars[col_ref_chuva] > 0
mask_sem_chuva = df_vars[col_ref_chuva] == 0

df_com = calcular_spearman_subconjunto(
    s_lagoa, df_vars, mask_com_chuva
).rename(columns={'rho': 'rho_com_chuva', 'n_pares': 'n_com_chuva'})
df_sem = calcular_spearman_subconjunto(
    s_lagoa, df_vars, mask_sem_chuva
).rename(columns={'rho': 'rho_sem_chuva', 'n_pares': 'n_sem_chuva'})

df_regimes = df_com[['variavel', 'n_com_chuva', 'rho_com_chuva']].merge(
    df_sem[['variavel', 'n_sem_chuva', 'rho_sem_chuva']],
    on='variavel',
    how='outer',
)
df_regimes['diff_com_menos_sem'] = (
    df_regimes['rho_com_chuva'] - df_regimes['rho_sem_chuva']
).round(4)
df_regimes = df_regimes.sort_values(
    by='rho_com_chuva', key=lambda s: s.abs(), ascending=False
)

caminho_regimes_csv = os.path.join(
    pasta_eda, 'spearman_regimes_chuva_detalhado.csv'
)
df_regimes.to_csv(caminho_regimes_csv, index=False)

print(f'=== SPEARMAN POR REGIME ({col_ref_chuva} > 0 vs. == 0) ===')
print(df_regimes.head(15).to_string(index=False))

# 4. Defasagens adicionais da maré (-6h a +12h a cada 30 min + grade fina de 4h a 8h a cada 10 min)
col_mare = (
    'mare_10min_m'
    if 'mare_10min_m' in df_vars.columns
    else (
        'nivel_mar_total_m' if 'nivel_mar_total_m' in df_vars.columns else None
    )
)

df_lags_mare = pd.DataFrame()
if col_mare is not None:
  # shift > 0: maré ocorrida há k passos (antecede NA_lagoa)
  passos_amplo = list(range(-36, 73, 3))  # -6h (-360 min) até +12h (+720 min)
  passos_fino = list(range(24, 49, 1))  # +4h (+240 min) até +8h (+480 min) a cada 10 min
  passos_total = sorted(set(passos_amplo + passos_fino))

  linhas_lags = []
  for p in passos_total:
    s_m_shift = df_vars[col_mare].shift(p)
    par = pd.concat([s_lagoa, s_m_shift], axis=1).dropna()
    if len(par) > 2:
      rho, p_val = spearmanr(par.iloc[:, 0], par.iloc[:, 1])
    else:
      rho, p_val = np.nan, np.nan
    linhas_lags.append({
        'lag_passos_10min': p,
        'lag_minutos': p * 10,
        'lag_horas': round(p * 10 / 60.0, 2),
        'n_pares': len(par),
        'rho': round(float(rho), 4) if pd.notna(rho) else np.nan,
        'p_valor': p_val,
    })

  df_lags_mare = pd.DataFrame(linhas_lags).sort_values('lag_minutos')
  caminho_lags_csv = os.path.join(
      pasta_eda, 'spearman_lags_mare_estendido.csv'
  )
  df_lags_mare.to_csv(caminho_lags_csv, index=False)

  melhor_lag = df_lags_mare.loc[df_lags_mare['rho'].abs().idxmax()]
  print('\n=== PICO DE SPEARMAN NAS DEFASAGENS DA MARÉ ===')
  print(
      f"  • Maior |ρ|: {melhor_lag['rho']:+.4f} em lag ="
      f" {int(melhor_lag['lag_minutos'])} min ({melhor_lag['lag_horas']} h)"
  )

# 5. Gráficos
plt.rcParams['figure.dpi'] = 140

# 5a. Comparação Com chuva vs. Sem chuva (top 12 variáveis)
top_reg = df_regimes.dropna(subset=['rho_com_chuva', 'rho_sem_chuva']).head(12)
if not top_reg.empty:
  top_plot = top_reg.iloc[::-1]
  y_pos = np.arange(len(top_plot))
  altura = 0.36

  fig, ax = plt.subplots(figsize=(11, 5.5))
  ax.barh(
      y_pos + altura / 2,
      top_plot['rho_com_chuva'],
      height=altura,
      color='#1f77b4',
      label=f'Com chuva ({col_ref_chuva} > 0)',
  )
  ax.barh(
      y_pos - altura / 2,
      top_plot['rho_sem_chuva'],
      height=altura,
      color='#7fcdbb',
      edgecolor='#238b45',
      linewidth=0.6,
      label=f'Sem chuva ({col_ref_chuva} == 0)',
  )
  ax.axvline(0, color='black', linewidth=0.9)
  ax.set_yticks(y_pos)
  ax.set_yticklabels(top_plot['variavel'])
  ax.set_xlabel('Correlação de Spearman (ρ) com NA_lagoa')
  ax.set_title('Spearman por Regime de Chuva: Com Chuva vs. Sem Chuva')
  ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
  ax.grid(True, axis='x', alpha=0.25)
  plt.tight_layout()
  caminho_fig_reg = os.path.join(
      pasta_eda, 'spearman_regimes_com_vs_sem_chuva.png'
  )
  plt.savefig(caminho_fig_reg, bbox_inches='tight')
  plt.close()

# 5b. Curva de defasagens da maré (-6h a +12h + detalhe em horas)
if not df_lags_mare.empty:
  fig, ax = plt.subplots(figsize=(11.5, 4.6))
  ax.plot(
      df_lags_mare['lag_horas'],
      df_lags_mare['rho'],
      marker='o',
      markersize=3.5,
      color='#cb181d',
      linewidth=1.5,
      label='Spearman (ρ) maré defasada vs. NA_lagoa',
  )
  ax.axvline(
      6.0,
      color='#08519c',
      linestyle='--',
      linewidth=1.1,
      label='6h (360 min)',
  )
  ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.9)
  ax.axhline(0.0, color='black', linewidth=0.8)
  ax.scatter(
      [melhor_lag['lag_horas']],
      [melhor_lag['rho']],
      color='#08519c',
      s=55,
      zorder=5,
      label=(
          f"Pico: {melhor_lag['rho']:+.3f} em"
          f" {melhor_lag['lag_horas']:.2f}h ({int(melhor_lag['lag_minutos'])}m)"
      ),
  )
  ax.set_xlabel('Defasagem da maré (horas; + = maré antecede NA_lagoa)')
  ax.set_ylabel('Correlação de Spearman (ρ)')
  ax.set_title('Spearman — Defasagens Estendidas da Maré (-6h a +12h)')
  ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
  ax.grid(True, alpha=0.25)
  plt.tight_layout()
  caminho_fig_lags = os.path.join(
      pasta_eda, 'spearman_lags_mare_estendido.png'
  )
  plt.savefig(caminho_fig_lags, bbox_inches='tight')
  plt.close()

print(f'\nArquivos salvos em: {pasta_eda}')

=== SPEARMAN POR REGIME (chuva_acum_6h_mm > 0 vs. == 0) ===
         variavel  n_com_chuva  rho_com_chuva  n_sem_chuva  rho_sem_chuva  diff_com_menos_sem
chuva_acum_24h_mm        77303         0.3825       396196         0.3085              0.0740
  mare_lag_360min        77303         0.3490       396196         0.4818             -0.1328
chuva_acum_12h_mm        77303         0.2148       396196         0.1875              0.0273
 chuva_lag_360min        77303         0.1734       396196         0.0289              0.1445
 chuva_lag_180min        77303         0.1039       396196            NaN                 NaN
     mare_diff_1h        77303         0.0792       396196         0.0777              0.0015
 chuva_lag_120min        77303         0.0706       396196            NaN                 NaN
 chuva_acum_6h_mm        77303         0.0668       396196            NaN                 NaN
  mare_lag_180min        77303         0.0539       396196         0.1690             -0.1151
